# Task 4 – Improving Model Performance

**Agent:** Claude (claude-sonnet-4-6)  
**Date:** 2026-03-18  
**Dataset:** `data/processed/Claude_clean.csv`

---

## Objective

Build on the Task 3 baseline (HistGradientBoosting, default params) and improve classification performance through:

1. **Feature engineering** – service-rating aggregates, delay transforms, log-distance, key interaction term  
2. **Better categorical encoding** – ordinal encoding for `Class`, binary mapping for `Gender` / `Type of Travel` / `Customer Type`  
3. **Hyperparameter tuning** – `RandomizedSearchCV` (30 iterations, 5-fold stratified CV) scoring on ROC AUC  
4. **Model upgrade** – XGBoost (if available) or tuned HistGradientBoosting  
5. **Cross-validation** – 5-fold CV reported alongside held-out test-set metrics

---

### Task 3 Baseline (for reference)

| Model | Accuracy | F1 Score | ROC AUC |
|-------|----------|----------|---------|
| Logistic Regression | 0.8744 | 0.8529 | 0.9270 |
| Random Forest | 0.9596 | 0.9530 | 0.9933 |
| **HistGradientBoosting** | **0.9617** | **0.9552** | **0.9948** |

## Setup

> **Run from project root** so that relative paths (`data/`, `figures/`, `results/`) resolve correctly.

In [1]:
import os

# Ensure we are running from the project root
if os.path.basename(os.getcwd()) == 'task4':
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/sophieburk/Downloads/Predictive Analytics/Group Project/predictive_agent_benchmark-main


## Imports & Configuration

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, cross_val_score, StratifiedKFold
)
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    roc_curve, confusion_matrix, classification_report,
    ConfusionMatrixDisplay,
)
from scipy.stats import randint, uniform

warnings.filterwarnings('ignore')

# Try XGBoost; fall back gracefully
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
    print('XGBoost available: True')
except ImportError:
    HAS_XGBOOST = False
    print('XGBoost not available – using HistGradientBoostingClassifier')

# Figure output
FIGURES_DIR = 'figures/Claude'
os.makedirs(FIGURES_DIR, exist_ok=True)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
FIG_COUNTER = [11]   # Task 4 figures begin at fig_12

def save_and_show(fig, name):
    FIG_COUNTER[0] += 1
    filename = f"{FIGURES_DIR}/fig_{FIG_COUNTER[0]:02d}_{name}.png"
    fig.savefig(filename, dpi=150, bbox_inches='tight')
    print(f'Saved: {filename}')
    plt.show()
    plt.close(fig)

XGBoost available: True


## 1. Load Dataset

In [3]:
df = pd.read_csv('data/processed/Claude_clean.csv')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df = df.drop(columns=['id'])
df.head()

Loaded: 129,487 rows × 24 columns


,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,3,1,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,Male,disloyal Customer,25,Business travel,Business,235,3,2,3,3,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,Female,Loyal Customer,26,Business travel,Business,1142,2,2,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,Female,Loyal Customer,25,Business travel,Business,562,2,5,5,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,Male,Loyal Customer,61,Business travel,Business,214,3,3,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


## 2. Encode Target

In [4]:
TARGET = 'satisfaction'
le_target = LabelEncoder()
y_all = le_target.fit_transform(df[TARGET])
print(f'Target encoding: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}')

Target encoding: {'neutral or dissatisfied': np.int64(0), 'satisfied': np.int64(1)}


## 3. Feature Engineering

New features created:

| Feature | Description |
|---------|-------------|
| `service_avg` | Mean of 14 service-rating columns |
| `service_min` / `service_max` | Worst / best single service score |
| `service_std` | Variability across service ratings |
| `service_range` | `max − min` of service ratings |
| `top_ratings` | Count of ratings ≥ 4 |
| `poor_ratings` | Count of ratings ≤ 2 |
| `total_delay` | Departure + Arrival delay (minutes) |
| `is_delayed` | Binary flag: any delay |
| `log_total_delay` | `log1p(total_delay)` – compresses long tail |
| `log_distance` | `log1p(Flight Distance)` |
| `boarding_x_entertainment` | Online boarding × Inflight entertainment |
| `age_group` | Ordinal age bins: Young/Adult/MiddleAge/Senior |

In [5]:
SERVICE_COLS = [
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Inflight service', 'Cleanliness',
]

X = df.drop(columns=[TARGET]).copy()

# Service-rating aggregates
X['service_avg']   = X[SERVICE_COLS].mean(axis=1)
X['service_min']   = X[SERVICE_COLS].min(axis=1)
X['service_max']   = X[SERVICE_COLS].max(axis=1)
X['service_std']   = X[SERVICE_COLS].std(axis=1)
X['service_range'] = X['service_max'] - X['service_min']
X['top_ratings']   = (X[SERVICE_COLS] >= 4).sum(axis=1)
X['poor_ratings']  = (X[SERVICE_COLS] <= 2).sum(axis=1)

# Delay features
X['total_delay']     = X['Departure Delay in Minutes'] + X['Arrival Delay in Minutes']
X['is_delayed']      = (X['total_delay'] > 0).astype(int)
X['log_total_delay'] = np.log1p(X['total_delay'])

# Distance
X['log_distance'] = np.log1p(X['Flight Distance'])

# Interaction
X['boarding_x_entertainment'] = X['Online boarding'] * X['Inflight entertainment']

# Age group
X['age_group'] = pd.cut(
    X['Age'], bins=[0, 25, 40, 60, 100], labels=[0, 1, 2, 3]
).astype(float)

print(f'Feature count after engineering: {X.shape[1]}')

Feature count after engineering: 35


## 4. Categorical Encoding

- **Class** → ordinal: Eco=0, Eco Plus=1, Business=2  
- **Gender** → binary: Male=1, Female=0  
- **Customer Type** → binary: Loyal=1, Disloyal=0  
- **Type of Travel** → binary: Business travel=1, Personal Travel=0

In [6]:
X['Class'] = OrdinalEncoder(
    categories=[['Eco', 'Eco Plus', 'Business']]
).fit_transform(X[['Class']]).flatten()

X['Gender']         = X['Gender'].map({'Male': 1, 'Female': 0})
X['Customer Type']  = X['Customer Type'].map({'Loyal Customer': 1, 'disloyal Customer': 0})
X['Type of Travel'] = X['Type of Travel'].map({'Business travel': 1, 'Personal Travel': 0})

print(f'Final feature matrix: {X.shape}')
print(f'dtypes: {X.dtypes.value_counts().to_dict()}')
X.describe().round(3)

Final feature matrix: (129487, 35)
dtypes: {dtype('int64'): 27, dtype('float64'): 8}


,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,...,service_std,service_range,top_ratings,poor_ratings,total_delay,is_delayed,log_total_delay,log_distance,boarding_x_entertainment,age_group
count,129487.000,129487.000,129487.000,129487.000,129487.000,129487.000,129487.000,129487.000,129487.000,129487.000,...,129487.000,129487.000,129487.000,129487.000,129487.000,129487.000,129487.000,129487.000,129487.000,129487.000
mean,0.493,0.817,39.429,0.691,1.030,1190.211,2.729,3.057,2.757,2.977,...,1.162,3.299,6.734,4.235,29.735,0.542,1.679,6.706,11.435,1.343
std,0.500,0.387,15.118,0.462,0.963,997.561,1.329,1.527,1.402,1.279,...,0.352,0.899,3.417,3.023,75.733,0.498,1.851,0.916,7.234,0.902
min,0.000,0.000,7.000,0.000,0.000,31.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,3.466,0.000,0.000
25%,0.000,1.000,27.000,0.000,0.000,414.000,2.000,2.000,2.000,2.000,...,0.914,3.000,4.000,2.000,0.000,0.000,0.000,6.028,5.000,1.000
50%,0.000,1.000,40.000,1.000,1.000,844.000,3.000,3.000,3.000,3.000,...,1.151,3.000,7.000,4.000,2.000,1.000,1.099,6.739,10.000,1.000
75%,1.000,1.000,51.000,1.000,2.000,1744.000,4.000,4.000,4.000,4.000,...,1.406,4.000,9.000,6.000,24.000,1.000,3.219,7.465,16.000,2.000
max,1.000,1.000,85.000,1.000,2.000,4983.000,5.000,5.000,5.000,5.000,...,2.344,5.000,14.000,14.000,3176.000,1.000,8.064,8.514,25.000,3.000


## 5. Train / Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_all, test_size=0.2, random_state=42, stratify=y_all
)
print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')

Train: 103,589  |  Test: 25,898


## 6. Baseline Recap (Task 3 HistGradientBoosting, default params)

In [8]:
# Rebuild Task-3 feature set for a fair comparison
df_orig = pd.read_csv('data/processed/Claude_clean.csv').drop(columns=['id'])
y_orig  = le_target.transform(df_orig[TARGET])
X_orig  = df_orig.drop(columns=[TARGET])
X_orig_enc = pd.get_dummies(X_orig, columns=X_orig.select_dtypes('object').columns.tolist(), drop_first=True)

X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_orig_enc, y_orig, test_size=0.2, random_state=42, stratify=y_orig
)

baseline_model = HistGradientBoostingClassifier(
    max_iter=200, max_depth=6, learning_rate=0.1, random_state=42
)
baseline_model.fit(X_tr_b, y_tr_b)
y_pred_b = baseline_model.predict(X_te_b)
y_prob_b = baseline_model.predict_proba(X_te_b)[:, 1]
fpr_b, tpr_b, _ = roc_curve(y_te_b, y_prob_b)

baseline_results = {
    'Accuracy': accuracy_score(y_te_b, y_pred_b),
    'F1 Score': f1_score(y_te_b, y_pred_b),
    'ROC AUC':  roc_auc_score(y_te_b, y_prob_b),
    'fpr': fpr_b, 'tpr': tpr_b, 'y_pred': y_pred_b,
}
print(f'Baseline Accuracy : {baseline_results["Accuracy"]:.4f}')
print(f'Baseline F1 Score : {baseline_results["F1 Score"]:.4f}')
print(f'Baseline ROC AUC  : {baseline_results["ROC AUC"]:.4f}')

Baseline Accuracy : 0.9617
Baseline F1 Score : 0.9552
Baseline ROC AUC  : 0.9948


## 7. Hyperparameter Tuning – RandomizedSearchCV

- **Search space**: 30 random parameter combinations  
- **CV strategy**: 5-fold stratified  
- **Scoring**: ROC AUC

In [9]:
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

if HAS_XGBOOST:
    MODEL_LABEL = 'XGBoost (Tuned)'
    base_estimator = XGBClassifier(
        random_state=42, n_jobs=-1,
        eval_metric='logloss', use_label_encoder=False, tree_method='hist',
    )
    param_dist = {
        'n_estimators':     randint(200, 600),
        'max_depth':        randint(4, 10),
        'learning_rate':    uniform(0.02, 0.18),
        'subsample':        uniform(0.7, 0.3),
        'colsample_bytree': uniform(0.6, 0.4),
        'min_child_weight': randint(1, 10),
        'gamma':            uniform(0.0, 0.3),
        'reg_alpha':        uniform(0.0, 1.0),
        'reg_lambda':       uniform(0.5, 1.5),
    }
else:
    MODEL_LABEL = 'HistGradientBoosting (Tuned)'
    base_estimator = HistGradientBoostingClassifier(random_state=42)
    param_dist = {
        'max_iter':          randint(200, 600),
        'max_depth':         randint(4, 10),
        'learning_rate':     uniform(0.02, 0.18),
        'min_samples_leaf':  randint(10, 60),
        'max_leaf_nodes':    randint(20, 63),
        'l2_regularization': uniform(0.0, 1.0),
    }

search = RandomizedSearchCV(
    base_estimator,
    param_distributions=param_dist,
    n_iter=30,
    cv=cv_splitter,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

print(f'Running RandomizedSearchCV for {MODEL_LABEL}...')
search.fit(X_train, y_train)
print(f'\nBest CV ROC AUC : {search.best_score_:.4f}')
print(f'Best params     :')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

Running RandomizedSearchCV for XGBoost (Tuned)...
Fitting 5 folds for each of 30 candidates, totalling 150 fits


/Users/sophieburk/Downloads/Predictive Analytics/Group Project/predictive_agent_benchmark-main/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:36:32] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/sophieburk/Downloads/Predictive Analytics/Group Project/predictive_agent_benchmark-main/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:36:33] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/sophieburk/Downloads/Predictive Analytics/Group Project/predictive_agent_benchmark-main/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:36:33] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/sophieburk/Downloa


Best CV ROC AUC : 0.9953
Best params     :
  colsample_bytree: 0.9895022075365837
  gamma: 0.06983140212909127
  learning_rate: 0.036309158215907744
  max_depth: 9
  min_child_weight: 3
  n_estimators: 563
  reg_alpha: 0.5142344384136116
  reg_lambda: 1.3886218532930636
  subsample: 0.7139351238159992


## 8. Evaluate Improved Model on Test Set

In [10]:
best_model = search.best_estimator_
y_pred_t   = best_model.predict(X_test)
y_prob_t   = best_model.predict_proba(X_test)[:, 1]
fpr_t, tpr_t, _ = roc_curve(y_test, y_prob_t)

improved_results = {
    'Accuracy': accuracy_score(y_test, y_pred_t),
    'F1 Score': f1_score(y_test, y_pred_t),
    'ROC AUC':  roc_auc_score(y_test, y_prob_t),
    'fpr': fpr_t, 'tpr': tpr_t, 'y_pred': y_pred_t,
}

print(f'{MODEL_LABEL} – Test-set Results:')
print(f'  Accuracy : {improved_results["Accuracy"]:.4f}')
print(f'  F1 Score : {improved_results["F1 Score"]:.4f}')
print(f'  ROC AUC  : {improved_results["ROC AUC"]:.4f}')
print()
print(classification_report(y_test, y_pred_t, target_names=le_target.classes_))

XGBoost (Tuned) – Test-set Results:
  Accuracy : 0.9640
  F1 Score : 0.9580
  ROC AUC  : 0.9954

                         precision    recall  f1-score   support

neutral or dissatisfied       0.96      0.98      0.97     14645
              satisfied       0.97      0.94      0.96     11253

               accuracy                           0.96     25898
              macro avg       0.97      0.96      0.96     25898
           weighted avg       0.96      0.96      0.96     25898



## 9. Cross-Validation (5-fold)

In [11]:
cv_auc = cross_val_score(best_model, X_train, y_train, cv=cv_splitter, scoring='roc_auc',  n_jobs=-1)
cv_acc = cross_val_score(best_model, X_train, y_train, cv=cv_splitter, scoring='accuracy', n_jobs=-1)
cv_f1  = cross_val_score(best_model, X_train, y_train, cv=cv_splitter, scoring='f1',       n_jobs=-1)

print(f'CV ROC AUC : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')
print(f'CV Accuracy: {cv_acc.mean():.4f} ± {cv_acc.std():.4f}')
print(f'CV F1 Score: {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')

/Users/sophieburk/Downloads/Predictive Analytics/Group Project/predictive_agent_benchmark-main/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:39:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/sophieburk/Downloads/Predictive Analytics/Group Project/predictive_agent_benchmark-main/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:39:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/sophieburk/Downloads/Predictive Analytics/Group Project/predictive_agent_benchmark-main/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:39:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/sophieburk/Downloa

CV ROC AUC : 0.9953 ± 0.0002
CV Accuracy: 0.9637 ± 0.0012
CV F1 Score: 0.9576 ± 0.0014


## 10. Performance Comparison Table

In [13]:
metrics = ['Accuracy', 'F1 Score', 'ROC AUC']

comparison = pd.DataFrame({
    'Baseline (HistGB, default)': {m: baseline_results[m] for m in metrics},
    MODEL_LABEL:                  {m: improved_results[m] for m in metrics},
}).T

comparison['Δ Accuracy'] = comparison['Accuracy'] - comparison.loc['Baseline (HistGB, default)', 'Accuracy']
comparison['Δ F1 Score'] = comparison['F1 Score'] - comparison.loc['Baseline (HistGB, default)', 'F1 Score']
comparison['Δ ROC AUC']  = comparison['ROC AUC']  - comparison.loc['Baseline (HistGB, default)', 'ROC AUC']

comparison.style.format({
    'Accuracy': '{:.4f}', 'F1 Score': '{:.4f}', 'ROC AUC': '{:.4f}',
    'Δ Accuracy': '{:+.4f}', 'Δ F1 Score': '{:+.4f}', 'Δ ROC AUC': '{:+.4f}',
}).background_gradient(subset=['Accuracy', 'F1 Score', 'ROC AUC'], cmap='YlGn')

,Accuracy,F1 Score,ROC AUC,Δ Accuracy,Δ F1 Score,Δ ROC AUC
"Baseline (HistGB, default)",0.9617,0.9552,0.9948,+0.0000,+0.0000,+0.0000
XGBoost (Tuned),0.9640,0.9580,0.9954,+0.0023,+0.0028,+0.0006


## 11. Visualisations

### Fig 12 – Metric Comparison Bar Chart

In [14]:
x = np.arange(len(metrics))
width = 0.35
colors = ['#4C72B0', '#DD8452']

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2,
               [baseline_results[m] for m in metrics],
               width, label='Baseline (HistGB, default)', color=colors[0], edgecolor='white')
bars2 = ax.bar(x + width/2,
               [improved_results[m] for m in metrics],
               width, label=MODEL_LABEL, color=colors[1], edgecolor='white')

for bars, vals in [(bars1, [baseline_results[m] for m in metrics]),
                   (bars2, [improved_results[m] for m in metrics])]:
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0.92, 1.03)
ax.set_ylabel('Score')
ax.set_title('Fig 12 – Baseline vs Improved Model: Metric Comparison', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.3f}'))
sns.despine()
plt.tight_layout()
save_and_show(fig, 'baseline_vs_improved_metrics')

Saved: figures/Claude/fig_12_baseline_vs_improved_metrics.png


### Fig 13 – ROC Curves Comparison

In [15]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_b, tpr_b,
        label=f'Baseline HistGB (AUC = {baseline_results["ROC AUC"]:.4f})',
        color=colors[0], linewidth=2)
ax.plot(fpr_t, tpr_t,
        label=f'{MODEL_LABEL} (AUC = {improved_results["ROC AUC"]:.4f})',
        color=colors[1], linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Fig 13 – ROC Curves: Baseline vs Improved', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
sns.despine()
plt.tight_layout()
save_and_show(fig, 'roc_curves_comparison')

Saved: figures/Claude/fig_13_roc_curves_comparison.png


### Fig 14 – Confusion Matrices

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (label, cm) in zip(axes, [
    ('Baseline HistGB', confusion_matrix(y_te_b, y_pred_b)),
    (MODEL_LABEL,       confusion_matrix(y_test, y_pred_t)),
]):
    disp = ConfusionMatrixDisplay(cm, display_labels=le_target.classes_)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(label, fontweight='bold', fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
fig.suptitle('Fig 14 – Confusion Matrices: Baseline vs Improved', fontweight='bold', y=1.02)
plt.tight_layout()
save_and_show(fig, 'confusion_matrices_comparison')

Saved: figures/Claude/fig_14_confusion_matrices_comparison.png


### Fig 15 – Feature Importance (Improved Model)

In [17]:
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X.columns)
    top25 = importances.sort_values(ascending=False).head(25)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(top25.index[::-1], top25.values[::-1], color='#DD8452', edgecolor='white')
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Fig 15 – Top 25 Feature Importances ({MODEL_LABEL})', fontweight='bold')
    sns.despine()
    plt.tight_layout()
    save_and_show(fig, 'feature_importance_improved')
else:
    print('Feature importances not available for this model.')

Saved: figures/Claude/fig_15_feature_importance_improved.png


### Fig 16 – 5-Fold CV ROC AUC Distribution

In [18]:
folds = [f'Fold {i+1}' for i in range(len(cv_auc))]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(folds, cv_auc, color='#55A868', edgecolor='white', zorder=3)
ax.axhline(cv_auc.mean(), color='red', linestyle='--', linewidth=1.5,
           label=f'Mean AUC = {cv_auc.mean():.4f}')
ax.set_ylim(max(0.99, cv_auc.min() - 0.002), min(1.0, cv_auc.max() + 0.004))
ax.set_ylabel('ROC AUC')
ax.set_title(f'Fig 16 – 5-Fold CV ROC AUC ({MODEL_LABEL})', fontweight='bold')
ax.legend(fontsize=9)
ax.yaxis.grid(True, zorder=0)
for i, v in enumerate(cv_auc):
    ax.text(i, v + 0.0003, f'{v:.4f}', ha='center', va='bottom', fontsize=9)
sns.despine()
plt.tight_layout()
save_and_show(fig, 'cv_roc_auc_folds')

Saved: figures/Claude/fig_16_cv_roc_auc_folds.png


## Final Summary

In [19]:
delta_acc = improved_results['Accuracy'] - baseline_results['Accuracy']
delta_f1  = improved_results['F1 Score'] - baseline_results['F1 Score']
delta_auc = improved_results['ROC AUC']  - baseline_results['ROC AUC']

print('=' * 75)
print('FINAL SUMMARY')
print('=' * 75)
print(f'{"Model":<40} {"Accuracy":>10} {"F1 Score":>10} {"ROC AUC":>10}')
print('-' * 75)
print(f'{"Baseline (HistGB, default)":<40} '
      f'{baseline_results["Accuracy"]:>10.4f} '
      f'{baseline_results["F1 Score"]:>10.4f} '
      f'{baseline_results["ROC AUC"]:>10.4f}')
print(f'{MODEL_LABEL:<40} '
      f'{improved_results["Accuracy"]:>10.4f} '
      f'{improved_results["F1 Score"]:>10.4f} '
      f'{improved_results["ROC AUC"]:>10.4f}')
print('-' * 75)
print(f'{"Improvement (Δ)":<40} '
      f'{delta_acc:>+10.4f} '
      f'{delta_f1:>+10.4f} '
      f'{delta_auc:>+10.4f}')
print()
print(f'Cross-Validation ({MODEL_LABEL}):')
print(f'  CV ROC AUC : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')
print(f'  CV Accuracy: {cv_acc.mean():.4f} ± {cv_acc.std():.4f}')
print(f'  CV F1 Score: {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')

FINAL SUMMARY
Model                                      Accuracy   F1 Score    ROC AUC
---------------------------------------------------------------------------
Baseline (HistGB, default)                   0.9617     0.9552     0.9948
XGBoost (Tuned)                              0.9640     0.9580     0.9954
---------------------------------------------------------------------------
Improvement (Δ)                             +0.0023    +0.0028    +0.0006

Cross-Validation (XGBoost (Tuned)):
  CV ROC AUC : 0.9953 ± 0.0002
  CV Accuracy: 0.9637 ± 0.0012
  CV F1 Score: 0.9576 ± 0.0014
